In [1]:
!pip install evaluate rouge_score absl-py

  Using cached rouge_score-0.1.2.tar.gz (17 kB)
  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24986 sha256=d75bc635e04cabac63a144b8b77e736144f0255569337128e6c4e767875eae55
  Stored in directory: /home/teom142/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge_score


In [2]:
import numpy as np  # 수치 연산을 위한 라이브러리
from datasets import load_dataset  # 데이터셋 로드를 위한 Hugging Face의 datasets 모듈

# "argilla/news-summary" 데이터셋의 테스트 분할을 로드
news = load_dataset("argilla/news-summary", split="test")

# 데이터셋을 Pandas DataFrame으로 변환하고, 5000개 샘플을 무작위로 선택 (재현성을 위해 random_state=42 사용)
df = news.to_pandas().sample(5000, random_state=42)[["text", "prediction"]]

# "prediction" 열의 값이 리스트 형태이므로, 첫 번째 딕셔너리의 "text" 값을 추출
df["prediction"] = df["prediction"].map(lambda x: x[0]["text"])

# 데이터를 훈련(60%), 검증(20%), 테스트(20%) 세트로 분할
# frac=1로 전체 데이터를 셔플링하고, 지정된 비율로 분할
train, valid, test = np.split(
    df.sample(frac=1, random_state=42), [int(0.6 * len(df)), int(0.8 * len(df))]
)

# 데이터셋의 예시와 크기를 출력
print(f"Source News : {train.text.iloc[0][:200]}")  # 훈련 데이터 첫 번째 뉴스 텍스트의 처음 200자
print(f"Summarization : {train.prediction.iloc[0][:50]}")  # 해당 뉴스의 요약문 처음 50자
print(f"Training Data Size : {len(train)}")  # 훈련 데이터 크기
print(f"Validation Data Size : {len(valid)}")  # 검증 데이터 크기
print(f"Testing Data Size : {len(test)}")  # 테스트 데이터 크기

/home/teom142/.conda/envs/pytorch_study/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Generating test split: 100%|██████████| 20417/20417 [00:00<00:00, 53510.03 examples/s]


Source News : DANANG, Vietnam (Reuters) - Russian President Vladimir Putin said on Saturday he had a normal dialogue with U.S. leader Donald Trump at a summit in Vietnam, and described Trump as civil, well-educated
Summarization : Putin says had useful interaction with Trump at Vi
Training Data Size : 3000
Validation Data Size : 1000
Testing Data Size : 1000


In [3]:
# PyTorch 및 관련 모듈 임포트
import torch  # PyTorch 프레임워크
from transformers import BartTokenizer  # BART 모델용 토크나이저
from torch.utils.data import TensorDataset, DataLoader  # 데이터셋 및 데이터로더
from torch.utils.data import RandomSampler, SequentialSampler  # 데이터 샘플링 방식
from torch.nn.utils.rnn import pad_sequence  # 시퀀스 패딩 처리

# 데이터셋을 토큰화하고 TensorDataset으로 변환하는 함수 정의
def make_dataset(data, tokenizer, device):
    # 뉴스 텍스트를 토큰화 (최대 길이 1024, 패딩은 가장 긴 시퀀스 기준, 잘림 허용)
    tokenized = tokenizer(
        text=data.text.tolist(),
        padding="longest",
        truncation=True,
        return_tensors="pt",  # PyTorch 텐서로 반환
        max_length=1024
    )
    labels = []  # 요약문 라벨 리스트 초기화
    input_ids = tokenized["input_ids"].to(device)  # 입력 ID를 지정된 디바이스로 이동
    attention_mask = tokenized["attention_mask"].to(device)  # 어텐션 마스크를 디바이스로 이동
    # 각 요약문을 토큰화하여 labels 리스트에 추가
    for target in data.prediction:
        labels.append(tokenizer.encode(target, return_tensors="pt").squeeze())

    # 라벨 시퀀스를 동일한 길이로 패딩 (-100은 손실 계산에서 무시되는 값)
    labels = pad_sequence(labels, batch_first=True, padding_value=-100).to(device)

    # 입력 ID, 어텐션 마스크, 라벨을 포함한 TensorDataset 반환
    return TensorDataset(input_ids, attention_mask, labels)

# DataLoader를 생성하는 함수 정의
def get_datalodader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)  # 지정된 샘플러로 데이터 샘플링
    # DataLoader 생성 (배치 크기와 샘플러 지정)
    dataloader = DataLoader(dataset, sampler=data_sampler, batch_size=batch_size)
    return dataloader

# 하이퍼파라미터 설정
epochs = 5  # 훈련 에포크 수
batch_size = 8  # 배치 크기
device = "cuda" if torch.cuda.is_available() else "cpu"  # GPU 사용 가능 여부에 따라 디바이스 설정

# BART 토크나이저 로드 ("facebook/bart-base" 사전 학습 모델 사용)
tokenizer = BartTokenizer.from_pretrained(
    pretrained_model_name_or_path="facebook/bart-base"
)

# 훈련, 검증, 테스트 데이터셋과 데이터로더 생성
train_dataset = make_dataset(train, tokenizer, device)  # 훈련 데이터셋
train_dataloader = get_datalodader(train_dataset, RandomSampler, batch_size)  # 무작위 샘플링 데이터로더

valid_dataset = make_dataset(valid, tokenizer, device)  # 검증 데이터셋
valid_dataloader = get_datalodader(valid_dataset, SequentialSampler, batch_size)  # 순차 샘플링 데이터로더

test_dataset = make_dataset(test, tokenizer, device)  # 테스트 데이터셋
test_dataloader = get_datalodader(test_dataset, SequentialSampler, batch_size)  # 순차 샘플링 데이터로더

# 훈련 데이터셋의 첫 번째 샘플 출력 (입력 ID, 어텐션 마스크, 라벨 확인용)
print(train_dataset[0])

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
/home/teom142/.conda/envs/pytorch_study/lib/python3.10/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


(tensor([   0,  495, 1889,  ...,    1,    1,    1], device='cuda:0'), tensor([1, 1, 1,  ..., 0, 0, 0], device='cuda:0'), tensor([    0, 35891,   161,    56,  5616, 10405,    19,   140,    23,  5490,
         3564,     2,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,
         -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100,  -100],
       device='cuda:0'))


In [4]:
from torch import optim
from transformers import BartForConditionalGeneration

# BART 모델 로드 및 디바이스로 이동
model = BartForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path="facebook/bart-base"
).to(device)
# AdamW 최적화 알고리즘 설정 (학습률 5e-5, epsilon 1e-8)
optimizer = optim.AdamW(model.parameters(), lr=5e-5, eps=1e-8)

In [5]:
# 모델 구조 출력 (계층별 모듈 이름 확인)
for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("└", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("│  └", ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("│  │  └", sssub_name)

model
└ shared
└ encoder
│  └ embed_tokens
│  └ embed_positions
│  └ layers
│  │  └ 0
│  │  └ 1
│  │  └ 2
│  │  └ 3
│  │  └ 4
│  │  └ 5
│  └ layernorm_embedding
└ decoder
│  └ embed_tokens
│  └ embed_positions
│  └ layers
│  │  └ 0
│  │  └ 1
│  │  └ 2
│  │  └ 3
│  │  └ 4
│  │  └ 5
│  └ layernorm_embedding
lm_head


In [6]:
# ROUGE 점수 계산을 위한 모듈 임포트
import numpy as np
import evaluate  # 평가 지표 계산 라이브러리

# ROUGE-2 점수를 계산하는 함수 정의
def calc_rouge(preds, labels):
    preds = preds.argmax(axis=-1)  # 예측 logits에서 가장 높은 확률의 토큰 선택

    # 라벨에서 -100(패딩 값)을 토크나이저의 패드 토큰 ID로 대체
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    # 예측값과 라벨을 텍스트로 디코딩 (특수 토큰 제외)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # ROUGE-2 점수 계산
    rouge2 = rouge_score.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )
    return rouge2["rouge2"]

# 모델 훈련 함수 정의
def train(model, optimizer, dataloader):
    model.train()  # 모델을 훈련 모드로 설정
    train_loss = 0.0  # 훈련 손실 초기화

    # 데이터로더에서 배치 단위로 처리
    for input_ids, attention_mask, labels in dataloader:
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels  # 손실 계산을 위한 정답 라벨
        )
        loss = outputs.loss  # 모델 출력에서 손실 추출
        train_loss += loss.item()  # 배치 손실 누적
        optimizer.zero_grad()  # 기울기 초기화
        loss.backward()  # 역전파로 기울기 계산
        optimizer.step()  # 모델 파라미터 업데이트
    train_loss = train_loss / len(dataloader)  # 평균 손실 계산
    return train_loss

# 모델 평가 함수 정의
def evaluation(model, dataloader):
    with torch.no_grad():  # 기울기 계산 비활성화
        model.eval()  # 모델을 평가 모드로 설정
        val_loss, val_rouge = 0.0, 0.0  # 손실과 ROUGE 점수 초기화

        # 데이터로더에서 배치 단위로 처리
        for input_ids, attention_mask, labels in dataloader:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            logits = outputs.logits  # 모델의 예측 logits
            loss = outputs.loss  # 손실

            # CPU로 데이터 이동 후 NumPy 배열로 변환
            logits = logits.detach().cpu().numpy()
            label_ids = labels.to("cpu").numpy()
            rouge = calc_rouge(logits, label_ids)  # ROUGE 점수 계산
            val_loss += loss.item()  # 배치 손실 누적
            val_rouge += rouge  # 배치 ROUGE 점수 누적
    val_loss = val_loss / len(dataloader)  # 평균 손실
    val_rouge = val_rouge / len(dataloader)  # 평균 ROUGE 점수
    return val_loss, val_rouge

# ROUGE 평가 지표 로드
rouge_score = evaluate.load("rouge", tokenizer=tokenizer)

# 훈련 루프 설정
best_loss = 10000  # 최적 손실 초기값 (높게 설정)
for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)  # 훈련 수행
    val_loss, val_accuracy = evaluation(model, valid_dataloader)  # 검증 수행

    # 에포크별 결과 출력
    print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f} Val Rouge {val_accuracy:.4f}")
    
    # 검증 손실이 이전보다 낮으면 모델 저장
    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "../models/BartForConditionalGeneration.pt")
        print("Saved the model weights")

Epoch 1: Train Loss: 2.1610 Val Loss: 1.8737 Val Rouge 0.2613
Saved the model weights
Epoch 2: Train Loss: 1.6012 Val Loss: 1.8797 Val Rouge 0.2590
Epoch 3: Train Loss: 1.2433 Val Loss: 1.9944 Val Rouge 0.2470
Epoch 4: Train Loss: 1.0589 Val Loss: 2.0971 Val Rouge 0.2592
Epoch 5: Train Loss: 0.7315 Val Loss: 2.2355 Val Rouge 0.2396


In [7]:
# 최적 모델 로드 및 테스트 평가
model = BartForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path="facebook/bart-base"
).to(device)
model.load_state_dict(torch.load("../models/BartForConditionalGeneration.pt"))  # 저장된 가중치 로드

test_loss, test_rouge_score = evaluation(model, test_dataloader)  # 테스트 데이터로 평가
print(f"Test Loss : {test_loss:.4f}")  # 테스트 손실 출력
print(f"Test ROUGE-2 Score : {test_rouge_score:.4f}")  # 테스트 ROUGE-2 점수 출력

/home/teom142/.conda/envs/pytorch_study/lib/python3.10/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Test Loss : 1.8237
Test ROUGE-2 Score : 0.2687


In [8]:
# 요약 생성을 위한 파이프라인 설정
from transformers import pipeline

summarizer = pipeline(
    task="summarization",  # 작업: 텍스트 요약
    model=model,  # 훈련된 모델
    tokenizer=tokenizer,  # 토크나이저
    max_length=54,  # 생성 요약의 최대 길이
    device="cpu"  # CPU에서 실행
)

# 테스트 데이터의 처음 5개 샘플에 대해 요약 생성 및 비교
for index in range(5):
    news_text = test.text.iloc[index]  # 뉴스 원문
    summarization = test.prediction.iloc[index]  # 정답 요약문
    predicted_summarization = summarizer(news_text)[0]["summary_text"]  # 모델 예측 요약문
    print(f"정답 요약문 : {summarization}")
    print(f"모델 요약문 : {predicted_summarization}\n")

정답 요약문 : Clinton leads Trump by 4 points in Washington Post: ABC News poll
모델 요약문 : Clinton leads Trump by 4 points in Washington Post-ABC News poll

정답 요약문 : Democrats question independence of Trump Supreme Court nominee
모델 요약문 : U.S. senators raise questions about Gorsuch's independence

정답 요약문 : In push for Yemen aid, U.S. warned Saudis of threats in Congress
모델 요약문 : U.S. warns Saudi Arabia that humanitarian situation could constrain aid

정답 요약문 : Romanian ruling party leader investigated over 'criminal group'
모델 요약문 : Romanian anti-graft prosecutors probe party leader

정답 요약문 : Billionaire environmental activist Tom Steyer endorses Clinton
모델 요약문 : Environmental activist Steyer backs Clinton for U.S. president

